# Nielsen Beverage Retail — EDA
**Market:** Pacific Division xAOC &nbsp;|&nbsp; **Period:** Jan 2025 – Feb 2026

In [4]:
import glob
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

## 1. Load Data

In [ ]:
files = sorted(glob.glob("../data/bquxjob*.csv"))
print(f"Loading {len(files)} CSV files...")
for f in files:
    print(f"  {f}: {os.path.getsize(f)/1e6:.2f} MB")

df = pd.concat([pd.read_csv(f, low_memory=False) for f in files], ignore_index=True)
df["week_ending"] = pd.to_datetime(df["week_ending"])

total_mb = sum(os.path.getsize(f) for f in files) / 1e6
print(f"\nTotal size : {total_mb:.1f} MB")
print(f"Shape      : {df.shape[0]:,} rows × {df.shape[1]} columns")

## 2. Basic Structure

In [6]:
print("Column dtypes")
print("=" * 50)
print(df.dtypes.to_string())

Column dtypes
category                                                   object
niagara_personal_chars                                     object
niagara_personal_chars_alias                               object
l1                                                         object
l2                                                         object
markets                                                    object
generic_tag                                                object
retailers_name_tag                                         object
retailers_div_req                                         float64
item                                                       object
upc                                                         int64
brand                                                      object
package_material_substance                                 object
package_general_shape                                      object
flavor                                                     obj

In [7]:
nulls = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(1)
null_df = pd.DataFrame({"null_count": nulls, "null_pct": null_pct})
null_df = null_df[null_df.null_count > 0]

print("Missing values (columns with at least 1 null)")
display(null_df)

Missing values (columns with at least 1 null)


,null_count,null_pct
category,317,0.200
retailers_div_req,162965,100.000
package_material_substance,317,0.200
package_general_shape,317,0.200
flavor,317,0.200
base_size,317,0.200
outer_pack_size,317,0.200
product_size,317,0.200
total_size,317,0.200
bp_concat_flavor,317,0.200


In [8]:
print(f"Duplicate rows: {df.duplicated().sum():,}")

Duplicate rows: 0


## 3. Time Coverage

In [9]:
print(f"Date range  : {df['week_ending'].min().date()} → {df['week_ending'].max().date()}")
print(f"Unique weeks: {df['week_ending'].nunique()}")
print(f"Years       : {sorted(df['year'].unique())}")
print()
print("Weeks per year:")
display(df.groupby("year")["week_ending"].nunique().rename("weeks").to_frame())
print("Rows per year:")
display(df.groupby("year").size().rename("rows").to_frame())

Date range  : 2025-01-04 → 2026-02-21
Unique weeks: 60
Years       : [2025, 2026]

Weeks per year:


,weeks
year,
2025,52
2026,8


Rows per year:


,rows
year,
2025,141429
2026,21536


## 4. Unique Identifiers

In [10]:
print(f"Unique UPCs  : {df['upc'].nunique():,}")
print(f"Unique items : {df['item'].nunique():,}")
print(f"Unique brands: {df['brand'].nunique():,}")

Unique UPCs  : 3,769
Unique items : 3,468
Unique brands: 377


## 5. Categorical Value Counts

In [11]:
cat_cols = [
    "category", "l1", "l2", "markets", "brand", "pack_type",
    "package_material_substance", "package_general_shape", "flavor",
    "niagara_personal_chars_alias", "Holiday",
]
for col in cat_cols:
    print(f"\n{'─'*50}")
    print(f"{col}  (unique: {df[col].nunique()})")
    print(f"{'─'*50}")
    display(df[col].value_counts(dropna=False).head(10).to_frame())


──────────────────────────────────────────────────
category  (unique: 6)
──────────────────────────────────────────────────


,count
category,
WATER,41983
VALUE ADD WATER,40338
SPORT DRINKS,35650
HEALTH/NUTRITION SHAKES,27925
PERFORMANCE NUTRITION SHAKES,12764
MEAL REPLACEMENT SHAKES,3988
NaN,317



──────────────────────────────────────────────────
l1  (unique: 1)
──────────────────────────────────────────────────


,count
l1,
Census Divisions,162965



──────────────────────────────────────────────────
l2  (unique: 1)
──────────────────────────────────────────────────


...

,count
flavor,
UNFLAVORED,59300
VANILLA,5740
CHOCOLATE,5514
NOT APPLICABLE,5199
FRUIT PUNCH,3399
LEMON LIME,2566
STRAWBERRY,2542
ORANGE,2521
GRAPE,1651



──────────────────────────────────────────────────
niagara_personal_chars_alias  (unique: 4)
──────────────────────────────────────────────────


,count
niagara_personal_chars_alias,
PC_Total SUW,60457
PC_Protein Shakes,44783
PC_Sport Drinks,35724
PC_Total SFW,22001



──────────────────────────────────────────────────
Holiday  (unique: 14)
──────────────────────────────────────────────────


,count
Holiday,
No Holiday,122227
Valentine's Day & Washington's Birthday,5394
"Martin Luther King, Jr. Day",5342
Juneteenth,2805
Memorial Day,2782
Eastern Easter & Western Easter,2770
Labor Day,2737
4th of July,2726
Columbus Day,2719


## 6. Numeric Summary

In [12]:
num_cols = [
    "sales", "units", "eq", "aup", "median_baseprice", "tpr_discount",
    "prc_acv", "acv", "tdp", "avg_eq_price", "avg_of_items",
    "seasonality_index", "any_promo_prc_acv", "no_promo_prc_acv",
]
display(df[num_cols].describe().round(3))

,sales,units,eq,aup,median_baseprice,tpr_discount,prc_acv,acv,tdp,avg_eq_price,avg_of_items,seasonality_index,any_promo_prc_acv,no_promo_prc_acv
count,162965.000,162965.000,162965.000,145154.000,162583.000,145154.000,145154.000,145154.000,145548.000,145154.000,145153.000,162965.000,145154.000,145154.000
mean,4172.065,1166.204,2710.362,6.672,7.274,5.029,9.174,9.174,9.149,11.294,1.000,1.000,3.311,5.871
std,15237.553,5044.644,13878.151,7.677,8.613,10.414,16.243,16.243,16.228,16.068,0.000,0.121,7.635,10.268
min,0.000,0.000,0.000,0.000,0.010,0.000,0.000,0.000,0.000,0.000,0.996,0.692,0.000,0.000
25%,21.854,5.417,2.713,2.066,2.240,0.000,0.224,0.224,0.218,1.969,1.000,0.912,0.000,0.129
50%,346.789,76.706,60.891,3.595,3.808,0.000,1.764,1.764,1.747,7.860,1.000,0.991,0.159,1.108
75%,2397.784,542.063,727.633,8.442,8.980,6.150,10.062,10.062,10.025,16.594,1.000,1.065,2.243,6.799
max,540844.206,213949.407,1551476.512,160.725,160.725,100.000,93.738,93.738,93.738,1826.420,1.004,1.386,77.573,82.744


## 7. Sales Summary by Category

In [13]:
cat_sales = df.groupby("category").agg(
    total_sales=("sales", "sum"),
    total_units=("units", "sum"),
    avg_aup=("aup", "mean"),
    median_aup=("aup", "median"),
    n_rows=("sales", "count"),
    unique_brands=("brand", "nunique"),
    unique_upcs=("upc", "nunique"),
).sort_values("total_sales", ascending=False)
cat_sales[["avg_aup", "median_aup"]] = cat_sales[["avg_aup", "median_aup"]].round(2)
display(cat_sales.style.format({"total_sales": "${:,.0f}", "total_units": "{:,.0f}"}))

,total_sales,total_units,avg_aup,median_aup,n_rows,unique_brands,unique_upcs
category,,,,,,,
WATER,"$305,274,900","94,737,252",4.760000,2.980000,41983,110,937
SPORT DRINKS,"$161,841,506","51,192,410",4.590000,2.620000,35650,52,828
VALUE ADD WATER,"$98,451,629","30,468,963",5.320000,2.730000,40338,153,961
PERFORMANCE NUTRITION SHAKES,"$56,796,373","8,084,441",12.270000,8.570000,12764,31,306
HEALTH/NUTRITION SHAKES,"$54,580,336","5,294,193",10.970000,8.310000,27925,86,622
MEAL REPLACEMENT SHAKES,"$2,955,825","273,119",10.700000,9.400000,3988,14,105


## 8. Top Brands & Brand Concentration

In [14]:
brand_sales = (
    df.groupby("brand")["sales"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)
brand_sales.columns = ["brand", "total_sales"]
display(brand_sales.style.format({"total_sales": "${:,.0f}"}))

,brand,total_sales
0,PRIVATE LABEL,"$167,741,579"
1,GATORADE (THE GATORADE COMPANY),"$101,606,584"
2,CRYSTAL GEYSER (CG ROXANE LLC),"$45,122,521"
3,PREMIER (PREMIER NUTRITION),"$26,937,747"
4,ARROWHEAD (PRIMO BRANDS CORPORATION),"$26,854,212"
5,"GLACEAU SMART WATER (ENERGY BRANDS, INC.)","$22,177,635"
6,"GLACEAU VITAMIN WATER (ENERGY BRANDS, INC.)","$21,793,200"
7,CORE POWER (FAIRLIFE LLC),"$21,673,081"
8,ENSURE (ABBOTT NUTRITION),"$21,035,372"
9,PURE LIFE (PRIMO BRANDS CORPORATION),"$16,890,281"


In [15]:
all_brand_sales = df.groupby("brand")["sales"].sum().sort_values(ascending=False)
total_sales = all_brand_sales.sum()
cumshare = (all_brand_sales / total_sales).cumsum()
print(f"Top  5 brands: {cumshare.iloc[4]*100:.1f}% of total sales")
print(f"Top 10 brands: {cumshare.iloc[9]*100:.1f}% of total sales")
print(f"Top 20 brands: {cumshare.iloc[19]*100:.1f}% of total sales")
print(f"Brands needed for 80% of sales: {(cumshare < 0.8).sum() + 1}")

Top  5 brands: 54.2% of total sales
Top 10 brands: 69.4% of total sales
Top 20 brands: 85.6% of total sales
Brands needed for 80% of sales: 16


## 9. Pricing & Promotions

In [16]:
print("AUP (avg unit price) by category")
display(df.groupby("category")["aup"].describe().round(2))

AUP (avg unit price) by category


,count,mean,std,min,25%,50%,75%,max
category,,,,,,,,
HEALTH/NUTRITION SHAKES,25592.000,10.970,9.700,0.000,3.990,8.310,13.730,98.370
MEAL REPLACEMENT SHAKES,3317.000,10.700,6.160,0.010,7.140,9.400,13.980,44.990
PERFORMANCE NUTRITION SHAKES,11258.000,12.270,11.970,0.010,3.660,8.570,16.660,68.970
SPORT DRINKS,31899.000,4.590,4.390,0.000,1.990,2.620,6.540,53.990
VALUE ADD WATER,35787.000,5.320,5.790,0.000,1.860,2.730,6.720,61.300
WATER,37300.000,4.760,6.060,0.000,1.590,2.980,5.320,160.730


In [17]:
print("TPR Discount (%, promo rows only) by category")
display(df[df["tpr_discount"] > 0].groupby("category")["tpr_discount"].describe().round(2))

TPR Discount (%, promo rows only) by category


,count,mean,std,min,25%,50%,75%,max
category,,,,,,,,
HEALTH/NUTRITION SHAKES,8849.000,14.160,13.170,3.000,5.780,10.040,17.520,100.000
MEAL REPLACEMENT SHAKES,1209.000,17.580,15.750,3.020,6.450,12.050,23.370,99.620
PERFORMANCE NUTRITION SHAKES,4305.000,14.740,13.560,3.000,5.640,10.000,18.570,99.620
SPORT DRINKS,12580.000,12.850,12.050,3.000,5.250,8.820,15.630,100.000
VALUE ADD WATER,13745.000,15.200,13.410,3.000,5.980,10.800,19.050,100.000
WATER,10431.000,14.310,13.900,3.000,5.470,9.530,17.560,100.000


In [18]:
print("Promo flag distribution by category")
display(df.groupby(["category", "promo_flag"]).size().unstack(fill_value=0))

Promo flag distribution by category


promo_flag,0,1
category,,
HEALTH/NUTRITION SHAKES,19076,8849
MEAL REPLACEMENT SHAKES,2779,1209
PERFORMANCE NUTRITION SHAKES,8459,4305
SPORT DRINKS,23070,12580
VALUE ADD WATER,26593,13745
WATER,31552,10431


## 10. Zero-Sales Rows

In [19]:
zero_sales = df[df["sales"] == 0]
print(f"Total zero-sales rows: {len(zero_sales):,}  ({len(zero_sales)/len(df)*100:.1f}%)")
print("\nBy category:")
display(zero_sales["category"].value_counts(dropna=False).to_frame())
print("By promo_flag:")
display(zero_sales["promo_flag"].value_counts().to_frame())

Total zero-sales rows: 17,835  (10.9%)

By category:


,count
category,
WATER,4687
VALUE ADD WATER,4560
SPORT DRINKS,3761
HEALTH/NUTRITION SHAKES,2334
PERFORMANCE NUTRITION SHAKES,1506
MEAL REPLACEMENT SHAKES,671
NaN,316


By promo_flag:


,count
promo_flag,
0,17816
1,19


## 11. Distribution Metrics (ACV / TDP)

In [20]:
print(f"acv == prc_acv always? {(df['acv'] == df['prc_acv']).sum() == df['acv'].notna().sum()}")
print("\nprc_acv stats:")
display(df["prc_acv"].describe().round(3).to_frame())

acv == prc_acv always? True

prc_acv stats:


,prc_acv
count,145154.000
mean,9.174
std,16.243
min,0.000
25%,0.224
50%,1.764
75%,10.062
max,93.738


## 12. Pack Type by Category

In [21]:
display(df.groupby(["category", "pack_type"]).size().unstack(fill_value=0))

pack_type,COMBINATION PACK,MULTI PACK,SINGLE PACK
category,,,
HEALTH/NUTRITION SHAKES,51,15282,12592
MEAL REPLACEMENT SHAKES,0,2908,1080
PERFORMANCE NUTRITION SHAKES,80,6662,6022
SPORT DRINKS,2914,8960,23776
VALUE ADD WATER,1914,12191,26233
WATER,0,17834,24149


## 13. Seasonality

In [22]:
print(f"Peak flag counts   : {df['seasonality_peak_flag'].value_counts().to_dict()}")
print(f"Rolling peak counts: {df['seasonality_rolling_peak_flag'].value_counts().to_dict()}")
print("\nSeasonality index by category:")
display(df.groupby("category")["seasonality_index"].describe().round(3))

Peak flag counts   : {0: 132553, 1: 30412}
Rolling peak counts: {0: 119586, 1: 43379}

Seasonality index by category:


,count,mean,std,min,25%,50%,75%,max
category,,,,,,,,
HEALTH/NUTRITION SHAKES,27925.000,1.004,0.058,0.692,0.972,1.011,1.054,1.105
MEAL REPLACEMENT SHAKES,3988.000,1.007,0.103,0.707,0.957,1.011,1.091,1.219
PERFORMANCE NUTRITION SHAKES,12764.000,1.005,0.084,0.692,0.970,1.011,1.060,1.127
SPORT DRINKS,35650.000,0.999,0.188,0.705,0.844,0.922,1.147,1.386
VALUE ADD WATER,40338.000,0.999,0.099,0.796,0.921,0.977,1.081,1.235
WATER,41983.000,0.998,0.109,0.796,0.910,0.955,1.089,1.235


## 14. Weekly Sales Trend by Category

In [23]:
weekly = df.groupby(["week_ending", "category"])["sales"].sum().unstack(fill_value=0)
print("Sampled every 8 weeks:")
display(weekly.iloc[::8].round(0).style.format("{:,.0f}"))

Sampled every 8 weeks:


category,HEALTH/NUTRITION SHAKES,MEAL REPLACEMENT SHAKES,PERFORMANCE NUTRITION SHAKES,SPORT DRINKS,VALUE ADD WATER,WATER
week_ending,,,,,,
2025-01-04 00:00:00,"794,917","52,541","762,570","2,207,921","1,450,381","4,577,959"
2025-03-01 00:00:00,"870,588","55,906","999,009","2,418,305","1,586,241","4,735,982"
2025-04-26 00:00:00,"857,871","48,843","939,261","2,503,826","1,598,749","4,868,637"
2025-06-21 00:00:00,"906,192","49,563","963,767","3,189,158","1,844,977","5,745,346"
2025-08-16 00:00:00,"965,211","48,608","952,044","3,605,837","1,946,376","5,977,258"
2025-10-11 00:00:00,"984,138","48,351","972,032","2,929,921","1,647,935","5,219,882"
2025-12-06 00:00:00,"851,985","45,755","820,597","2,056,529","1,407,310","4,509,163"
2026-01-31 00:00:00,"914,633","44,944","1,037,005","2,347,823","1,489,887","4,563,953"


## 15. Correlation Matrix

In [24]:
key_cols = [
    "sales", "units", "eq", "aup", "prc_acv", "tdp",
    "tpr_discount", "seasonality_index", "avg_eq_price",
]
display(df[key_cols].corr().round(3).style.background_gradient(cmap="coolwarm", vmin=-1, vmax=1))

,sales,units,eq,aup,prc_acv,tdp,tpr_discount,seasonality_index,avg_eq_price
sales,1.000000,0.816000,0.619000,0.009000,0.458000,0.458000,-0.070000,0.027000,-0.066000
units,0.816000,1.000000,0.415000,-0.099000,0.451000,0.451000,-0.043000,0.028000,-0.040000
eq,0.619000,0.415000,1.000000,0.162000,0.155000,0.156000,-0.056000,0.014000,-0.126000
aup,0.009000,-0.099000,0.162000,1.000000,-0.029000,-0.029000,-0.088000,0.006000,-0.077000
prc_acv,0.458000,0.451000,0.155000,-0.029000,1.000000,1.000000,-0.096000,0.013000,0.003000
tdp,0.458000,0.451000,0.156000,-0.029000,1.000000,1.000000,-0.096000,0.013000,0.003000
tpr_discount,-0.070000,-0.043000,-0.056000,-0.088000,-0.096000,-0.096000,1.000000,0.021000,-0.021000
seasonality_index,0.027000,0.028000,0.014000,0.006000,0.013000,0.013000,0.021000,1.000000,-0.017000
avg_eq_price,-0.066000,-0.040000,-0.126000,-0.077000,0.003000,0.003000,-0.021000,-0.017000,1.000000


## 16. Null Count per Row Distribution

In [25]:
null_per_row = df.isnull().sum(axis=1)
print(f"0 nulls   : {(null_per_row == 0).sum():,}")
print(f"1–5 nulls : {((null_per_row >= 1) & (null_per_row <= 5)).sum():,}")
print(f"6–20 nulls: {((null_per_row >= 6) & (null_per_row <= 20)).sum():,}")
print(f">20 nulls : {(null_per_row > 20).sum():,}")

0 nulls   : 0
1–5 nulls : 84,553
6–20 nulls: 60,796
>20 nulls : 17,616


## 17. Sales Outliers (> 99.9th Percentile)

In [26]:
p999 = df["sales"].quantile(0.999)
outliers = (
    df[df["sales"] > p999][["category", "brand", "item", "week_ending", "sales", "units", "aup"]]
    .sort_values("sales", ascending=False)
    .head(10)
)
display(outliers.style.format({"sales": "${:,.0f}", "units": "{:,.0f}", "aup": "${:.2f}"}))

,category,brand,item,week_ending,sales,units,aup
110381,WATER,PRIVATE LABEL,CTL BR DRNK WTR BTL 40PK 16.9 FL OZ,2025-07-05 00:00:00,"$540,844","109,846",$4.92
161901,WATER,PRIVATE LABEL,CTL BR DRNK WTR BTL 40PK 16.9 FL OZ,2025-09-06 00:00:00,"$540,303","99,590",$5.43
153696,WATER,PRIVATE LABEL,CTL BR DRNK WTR BTL 40PK 16.9 FL OZ,2025-08-16 00:00:00,"$537,339","102,471",$5.24
84078,WATER,PRIVATE LABEL,CTL BR DRNK WTR BTL 40PK 16.9 FL OZ,2025-06-14 00:00:00,"$536,846","98,199",$5.47
150956,WATER,PRIVATE LABEL,CTL BR DRNK WTR BTL 40PK 16.9 FL OZ,2025-08-09 00:00:00,"$528,769","107,479",$4.92
17320,WATER,PRIVATE LABEL,CTL BR DRNK WTR BTL 40PK 16.9 FL OZ,2025-06-07 00:00:00,"$517,791","94,680",$5.47
113100,WATER,PRIVATE LABEL,CTL BR DRNK WTR BTL 40PK 16.9 FL OZ,2025-07-12 00:00:00,"$516,263","104,902",$4.92
74640,WATER,PRIVATE LABEL,CTL BR DRNK WTR BTL 40PK 16.9 FL OZ,2025-09-13 00:00:00,"$510,975","94,103",$5.43
156436,WATER,PRIVATE LABEL,CTL BR DRNK WTR BTL 40PK 16.9 FL OZ,2025-08-23 00:00:00,"$509,105","93,351",$5.45
159172,WATER,PRIVATE LABEL,CTL BR DRNK WTR BTL 40PK 16.9 FL OZ,2025-08-30 00:00:00,"$508,184","93,206",$5.45
